# Lab 1 – Image Manipulation and Color Spaces

**Rješenja svih pet zadataka iz laboratorijske vježbe 1.**

Originalni zadatak: [unisb-rv-26-27/lab1](https://github.com/unisb-rv-26-27/lab1)

Pokreni ćelije redom (**Runtime → Run all**). Svaki zadatak zasebno prikazuje tražene rezultate. Komentari u kodu objašnjavaju glavne korake.


## Priprema – učitavanje biblioteka i slike

Koristimo **OpenCV** za obradu slike i **matplotlib** za prikaz. OpenCV čita sliku redoslijedom boja **BGR**, dok matplotlib očekuje **RGB**, pa je prilikom prikaza potrebna pretvorba.

Ako datoteka `images/peppers.png` nije dostupna u trenutnoj mapi, preuzet ćemo standardnu probnu sliku paprika iz javno dostupnog primjera. Ako već postoji izvorna slika iz predloška, ona se koristi bez zamjene.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import cv2
import numpy as np
import matplotlib.pyplot as plt

slika_putanja = Path("images/peppers.png")
slika_putanja.parent.mkdir(parents=True, exist_ok=True)

if not slika_putanja.is_file():
    url = "https://raw.githubusercontent.com/LingDong-/r1b/master/examples/assets/peppers.png"
    try:
        urlretrieve(url, str(slika_putanja))
    except Exception as e:
        raise RuntimeError(
            "Slika nije pronađena. Učitaj images/peppers.png u Colab ili provjeri internet vezu."
        ) from e

slika = cv2.imread(str(slika_putanja), cv2.IMREAD_COLOR)
if slika is None:
    raise FileNotFoundError(f"Ne mogu učitati sliku: {slika_putanja}")

print(f"Učitana slika: {slika_putanja} – {slika.shape[1]} × {slika.shape[0]} piksela")
plt.figure(figsize=(5, 5))
plt.imshow(cv2.cvtColor(slika, cv2.COLOR_BGR2RGB))
plt.title("Originalna slika")
plt.axis("off")
plt.show()

# 1. zadatak – prikaz triju kanala boje

Učitaj sliku i napravi **tri zasebne slike**: samo plavi, samo zeleni i samo crveni kanal. Na svakoj slici ostala dva kanala imaju vrijednost nula.


In [ ]:
# OpenCV koristi BGR raspored kanala: B=0, G=1, R=2
nazivi = ["Plavi kanal (B)", "Zeleni kanal (G)", "Crveni kanal (R)"]

fig, osi = plt.subplots(1, 3, figsize=(15, 5))
for kanal in range(3):
    samo_jedan_kanal = np.zeros_like(slika)
    samo_jedan_kanal[:, :, kanal] = slika[:, :, kanal]
    osi[kanal].imshow(cv2.cvtColor(samo_jedan_kanal, cv2.COLOR_BGR2RGB))
    osi[kanal].set_title(nazivi[kanal])
    osi[kanal].axis("off")

plt.tight_layout()
plt.show()

# 2. zadatak – okvir širine 10 piksela

Dodaj **crni okvir** širine 10 piksela na sve četiri strane slike i prikaži rezultat.


In [ ]:
# Gornji, donji, lijevi i desni okvir, svaki 10 piksela
slika_s_okvirom = cv2.copyMakeBorder(
    slika,
    10, 10, 10, 10,
    borderType=cv2.BORDER_CONSTANT,
    value=(0, 0, 0)  # crna boja u BGR formatu
)

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(slika_s_okvirom, cv2.COLOR_BGR2RGB))
plt.title("Slika s okvirom od 10 piksela")
plt.axis("off")
plt.show()

# 3. zadatak – smanjivanje broja redaka i stupaca

Napravi tri nove slike: prva ima **svaki drugi redak**, druga **svaki drugi stupac**, a treća **svaki drugi redak i stupac**. Na taj način ne koristimo interpolaciju, nego uzorkovanje svakog drugog piksela, prema uputi zadatka.


In [ ]:
pola_redaka = slika[::2, :, :]    # upola manje redaka (visina)
pola_stupaca = slika[:, ::2, :]   # upola manje stupaca (širina)
pola_oboje = slika[::2, ::2, :]  # upola manje redaka i stupaca

umanjene = [pola_redaka, pola_stupaca, pola_oboje]
naslovi = ["Pola redaka", "Pola stupaca", "Pola redaka i stupaca"]

fig, osi = plt.subplots(1, 3, figsize=(15, 5))
for os, nova_slika, naslov in zip(osi, umanjene, naslovi):
    os.imshow(cv2.cvtColor(nova_slika, cv2.COLOR_BGR2RGB))
    os.set_title(f"{naslov}\n{nova_slika.shape[1]} × {nova_slika.shape[0]}")
    os.axis("off")

plt.tight_layout()
plt.show()

# 4. zadatak – HSV: povećavanje H kanala za 30

Pretvori sliku `images/peppers.png` iz BGR u HSV. Dodaj **30** svakom elementu H kanala, pretvori je nazad u RGB i prikaži rezultat.

OpenCV kod 8-bitnih HSV slika koristi H raspon **0–179**. Izraz `% 180` osigurava ispravan povratak na početak raspona (npr. 170 + 30 → 20), bez prelijevanja 8-bitnog podatka.


In [ ]:
hsv = cv2.cvtColor(slika, cv2.COLOR_BGR2HSV)
hsv_plus_30 = hsv.copy()

# Privremeno pretvori H u širi cjelobrojni tip radi sigurnog zbrajanja.
hsv_plus_30[:, :, 0] = (
    (hsv_plus_30[:, :, 0].astype(np.int16) + 30) % 180
).astype(np.uint8)

rgb_plus_30 = cv2.cvtColor(hsv_plus_30, cv2.COLOR_HSV2RGB)

plt.figure(figsize=(6, 6))
plt.imshow(rgb_plus_30)
plt.title("HSV – H kanal povećan za 30")
plt.axis("off")
plt.show()

# 5. zadatak – HSV: postavljanje H kanala na 0

Ponovno učitaj izvornu sliku u HSV prostoru i postavi **sve vrijednosti H kanala na nulu**, bez mijenjanja zasićenja (S) i svjetline (V). Pretvori nazad u RGB i prikaži rezultat.


In [ ]:
hsv_nula = cv2.cvtColor(slika, cv2.COLOR_BGR2HSV)
hsv_nula[:, :, 0] = 0  # svim pikselima postavljamo H = 0

rgb_h_nula = cv2.cvtColor(hsv_nula, cv2.COLOR_HSV2RGB)

plt.figure(figsize=(6, 6))
plt.imshow(rgb_h_nula)
plt.title("HSV – H kanal postavljen na 0")
plt.axis("off")
plt.show()

---

## Predaja

Pokreni **Runtime → Run all** i provjeri da se svih pet rezultata prikazuje. Zatim u Google Colabu odaberi **File → Save a copy in GitHub** i spremi `lab1.ipynb` u **svoj GitHub Classroom repozitorij** (ne u javni repozitorij sa zadatkom).

Ako već postoji istoimena datoteka, odaberi zamjenu da u repozitoriju ostane rješenje zadatka.
